In [2]:
!pip install pandas numpy scikit-learn matplotlib seaborn flask ultralytics opencv-python streamlit

Defaulting to user installation because normal site-packages is not writeable
  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached seaborn-0.13.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached ultralytics-8.4.135-py3-none-any.whl.metadata (45 kB)
  Using cached opencv_python-5.0.0.93-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached streamlit-1.62.0-py3-none-any.whl.metadata (10 kB)
  Using cached scipy-1.18.1-cp314-cp314-win_amd64.whl.metadata (61 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached narwhals-2.25.0-py3-none-any.whl.metadata (15 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached blinker-1.9.0-py3-none-any.whl.metadata (1.6 kB)
  Using cached click-8.5.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached itsdangerous-2.2.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached jinja2-3.1.6-py3-none-any.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import cv2
import flask
import streamlit
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\ujwal\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


In [ ]:
df = pd.read_csv('dataset_traffic_accident_prediction1.csv')  
print(df.shape) 
print(df.columns.tolist()) 
df.head() 

(840, 14)
['Weather', 'Road_Type', 'Time_of_Day', 'Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Alcohol', 'Accident_Severity', 'Road_Condition', 'Vehicle_Type', 'Driver_Age', 'Driver_Experience', 'Road_Light_Condition', 'Accident']


,Weather,Road_Type,Time_of_Day,Traffic_Density,Speed_Limit,Number_of_Vehicles,Driver_Alcohol,Accident_Severity,Road_Condition,Vehicle_Type,Driver_Age,Driver_Experience,Road_Light_Condition,Accident
0,Rainy,City Road,Morning,1.0,100.0,5.0,0.0,NaN,Wet,Car,51.0,48.0,Artificial Light,0.0
1,Clear,Rural Road,Night,NaN,120.0,3.0,0.0,Moderate,Wet,Truck,49.0,43.0,Artificial Light,0.0
2,Rainy,Highway,Evening,1.0,60.0,4.0,0.0,Low,Icy,Car,54.0,52.0,Artificial Light,0.0
3,Clear,City Road,Afternoon,2.0,60.0,3.0,0.0,Low,Under Construction,Bus,34.0,31.0,Daylight,0.0
4,Rainy,Highway,Morning,1.0,195.0,11.0,0.0,Low,Dry,Car,62.0,55.0,Artificial Light,1.0


In [ ]:
print(df['Accident'].value_counts()) 
print(df['Accident'].value_counts(normalize=True) * 100) 

Accident
0.0    559
1.0    239
Name: count, dtype: int64
Accident
0.0    70.050125
1.0    29.949875
Name: proportion, dtype: float64


In [ ]:
print(df.isnull().sum()) 

Weather                 42
Road_Type               42
Time_of_Day             42
Traffic_Density         42
Speed_Limit             42
Number_of_Vehicles      42
Driver_Alcohol          42
Accident_Severity       42
Road_Condition          42
Vehicle_Type            42
Driver_Age              42
Driver_Experience       42
Road_Light_Condition    42
Accident                42
dtype: int64


In [ ]:
print(df[df.isnull().all(axis=1)].shape) 

(0, 14)


In [8]:
print(df[df.isnull().any(axis=1)].shape)

(435, 14)


In [9]:
df = df.dropna(subset=['Accident'])
print(df.shape)

(798, 14)


In [10]:
print(df.isnull().sum())

Weather                 40
Road_Type               40
Time_of_Day             38
Traffic_Density         40
Speed_Limit             41
Number_of_Vehicles      38
Driver_Alcohol          40
Accident_Severity       42
Road_Condition          41
Vehicle_Type            39
Driver_Age              40
Driver_Experience       41
Road_Light_Condition    40
Accident                 0
dtype: int64


In [11]:
# Numeric columns → fill missing values with median
numeric_cols = ['Traffic_Density', 'Speed_Limit', 'Number_of_Vehicles', 'Driver_Age', 'Driver_Experience']
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns → fill missing values with mode (most frequent value)
categorical_cols = ['Weather', 'Road_Type', 'Time_of_Day', 'Driver_Alcohol', 'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition']
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print(df.isnull().sum())

Weather                  0
Road_Type                0
Time_of_Day              0
Traffic_Density          0
Speed_Limit              0
Number_of_Vehicles       0
Driver_Alcohol           0
Accident_Severity       42
Road_Condition           0
Vehicle_Type             0
Driver_Age               0
Driver_Experience        0
Road_Light_Condition     0
Accident                 0
dtype: int64


In [12]:
categorical_cols = ['Weather', 'Road_Type', 'Time_of_Day', 'Driver_Alcohol', 'Road_Condition', 'Vehicle_Type', 'Road_Light_Condition']

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)
print(df_encoded.shape)
df_encoded.head()

(798, 26)


,Traffic_Density,Speed_Limit,Number_of_Vehicles,Accident_Severity,Driver_Age,Driver_Experience,Accident,Weather_Foggy,Weather_Rainy,Weather_Snowy,...,Time_of_Day_Night,Driver_Alcohol_1.0,Road_Condition_Icy,Road_Condition_Under Construction,Road_Condition_Wet,Vehicle_Type_Car,Vehicle_Type_Motorcycle,Vehicle_Type_Truck,Road_Light_Condition_Daylight,Road_Light_Condition_No Light
0,1.0,100.0,5.0,NaN,51.0,48.0,0.0,False,True,False,...,False,False,False,False,True,True,False,False,False,False
1,1.0,120.0,3.0,Moderate,49.0,43.0,0.0,False,False,False,...,True,False,False,False,True,False,False,True,False,False
2,1.0,60.0,4.0,Low,54.0,52.0,0.0,False,True,False,...,False,False,True,False,False,True,False,False,False,False
3,2.0,60.0,3.0,Low,34.0,31.0,0.0,False,False,False,...,False,False,False,True,False,False,False,False,True,False
4,1.0,195.0,11.0,Low,62.0,55.0,1.0,False,True,False,...,False,False,False,False,False,True,False,False,False,False
